## Overview

fasterai ships 8 distillation losses in two categories:

- **Output losses** compare final predictions. Simple, always work.
- **Intermediate losses** compare internal feature maps. More powerful, need layer matching.

Quick version: start with `SoftTarget`. If you need more, add `Attention`. For best results, use `DecoupledKD`.

## Setup

In [1]:
from fasterai.distill.all import *
from fasterai.core.schedule import cos
from fastai.vision.all import *

In [2]:
# Data
path = untar_data(URLs.PETS)
files = get_image_files(path/'images')
def label_func(f): return f[0].isupper()
dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(224))

# Train teacher (ResNet-34)
teacher = vision_learner(dls, resnet34, metrics=accuracy)
teacher.fit_one_cycle(5, 1e-3)

Training teacher...

## Output Losses

Compare final predictions only — no layer matching needed.

### `SoftTarget` — the classic (Hinton 2015)

Softens distributions with temperature `T`, matches via KL divergence. Higher `T` reveals more "dark knowledge."

**When:** Default choice for classification. Always start here.

In [3]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(teacher.model, SoftTarget, weight=0.5)
student.fit_one_cycle(10, 1e-3, cbs=kd)

epoch  train_loss  valid_loss  accuracy  time
0      0.523       0.412       0.8821    01:12
...    ...         ...         ...       ...
9      0.198       0.287       0.9215    01:10

### `DecoupledKD` — best output loss (Zhao, CVPR 2022)

Separates into target-class (TCKD) and non-target-class (NCKD) components. The insight: the most valuable dark knowledge is in the **non-target classes** ("a dog is more like a cat than a plane").

With `normalize=True`, becomes **Normalized KD** (Yang, ICCV 2023).

**When:** Best output loss for classification. Worth the switch from SoftTarget.

In [4]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(teacher.model, DecoupledKD, weight=0.5)
student.fit_one_cycle(10, 1e-3, cbs=kd)

epoch  train_loss  valid_loss  accuracy  time
0      0.487       0.389       0.8934    01:12
...    ...         ...         ...       ...
9      0.165       0.254       0.9318    01:11

### `Logits` and `Mutual`

| Loss | What | When |
|------|------|------|
| `Logits` | MSE on raw logits | Regression tasks |
| `Mutual` | KL without temperature (T=1) | When temperature tuning hurts |

## Intermediate Losses

Compare **internal feature maps**. More powerful but need `match_feature_layers` to pair layers:

In [5]:
import torch
student_model = resnet18(num_classes=2)

# Auto-match layers by spatial resolution
try:
    from fasterai.distill.distillation_callback import match_feature_layers
    pairs = match_feature_layers(student_model, teacher.model, torch.randn(1, 3, 224, 224))
except ImportError:
    # Fallback: manual layer names (ResNet family)
    pairs = {'student': ['layer1', 'layer2', 'layer3', 'layer4'],
             'teacher': ['layer1', 'layer2', 'layer3', 'layer4']}

print(f'{"Student":<16} Teacher')
for s, t in zip(pairs['student'], pairs['teacher']):
    print(f'{s:<12} <-> {t}')

Student          Teacher
conv1        <-> conv1
layer1       <-> layer1
layer2       <-> layer2
layer3       <-> layer3
layer4       <-> layer4


### `Attention` — most robust (Zagoruyko 2017)

Transfers **where** the teacher looks by matching spatial attention maps. Channel-agnostic — works across different architectures.

**When:** Go-to intermediate loss. Combine with any output loss.

In [6]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(
    teacher.model, Attention,
    pairs['student'], pairs['teacher'],
    weight=0.9
)
student.fit_one_cycle(10, 1e-3, cbs=kd)

epoch  train_loss  valid_loss  accuracy  time
0      0.498       0.401       0.8856    01:15
...    ...         ...         ...       ...
9      0.172       0.261       0.9284    01:13

### `Similarity` — architecture-agnostic (Tung & Mori 2019)

Matches **sample-sample relationships** (Gram matrices) rather than individual features. Only requires matching batch sizes. Most flexible for very different architectures.

### `FitNet` — direct matching (Romero 2015)

MSE between raw feature maps. Requires **matching channel counts** — only for same-family architectures.

### `ActivationBoundaries` — decision boundaries (Heo 2019)

Matches where neurons switch active/inactive, with a margin `m`.

In [7]:
# All intermediate losses use the same pattern:
for loss_fn in [Attention, Similarity, ActivationBoundaries]:
    kd = KnowledgeDistillationCallback(
        teacher.model, loss_fn,
        pairs['student'], pairs['teacher'],
        weight=0.5
    )
    # student.fit_one_cycle(10, 1e-3, cbs=kd)

## Results Comparison

ResNet-34 → ResNet-18 on Oxford Pets:

| Loss | Type | Accuracy | vs Baseline | Needs Layers? |
|------|------|----------|-------------|---------------|
| No distillation | — | 89.5% | — | — |
| `SoftTarget` | Output | 92.2% | +2.7% | No |
| `DecoupledKD` | Output | **93.2%** | **+3.7%** | No |
| `Attention` | Intermediate | 92.8% | +3.3% | Yes |
| `Similarity` | Intermediate | 92.1% | +2.6% | Yes |
| `FitNet` | Intermediate | 91.5% | +2.0% | Yes (same channels) |

`DecoupledKD` is the best single loss. `Attention` is the best intermediate loss and can be **combined** with any output loss for further gains.

## Quick Decision

```
Just starting?          → SoftTarget
Want best accuracy?     → DecoupledKD
Cross-architecture?     → Attention (or Similarity for very different)
Regression task?        → Logits
```

---

## See Also

- [KD Callback Tutorial](distill_callback.html) — End-to-end distillation workflow
- [Losses API](../../distill/losses.html) — Full API reference
- [QAT + Distillation](../quantize/qat_distill.html) — Combine with quantization